In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
zip_path = "/content/drive/MyDrive/early-fire-detection/zippy/data.zip"
!unzip -o -q {zip_path} -d /

In [3]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/early-fire-detection")
sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:
from helpers.utils import set_seed, make_run_dir, load_checkpoint, train_detection_run, save_run_config
from helpers.datahelperrr import FireDataset, collate_fn, evaluate_map

In [6]:
DATA_DIR = Path("/data")
RUNS = PROJECT_ROOT / "models" / "runs"
NUM_CLASSES = 3  # background=0, fire=1, smoke=2
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
def build_fasterrcnn(num_classes=NUM_CLASSES, trainable_backbone_layers=3):
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn_v2(weights=weights, trainable_backbone_layers=trainable_backbone_layers)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.transform.min_size = (640,)
    model.transform.max_size = 640
    return model

In [8]:
train_ds = FireDataset(DATA_DIR / "train", augment=True)
val_ds = FireDataset(DATA_DIR / "val", augment=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

In [ ]:
lr_sweep = [0.001, 0.005, 0.01, 0.02]
results = []

for lr in lr_sweep:
    lr_tag = str(lr).replace(".", "")
    set_seed(42)
    model = build_fasterrcnn(num_classes=NUM_CLASSES, trainable_backbone_layers=3).to(device)
    model.to(device)
    load_checkpoint(RUNS / "fasterrcnn_v2_freeze_head" / "weights" / "best.pt", model)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=0.0005)
    run_dir = make_run_dir(RUNS, f"fasterrcnn_v2_unfreeze_l3_lr{lr_tag}_hyper")
    save_run_config(run_dir, {
        "lr": lr,
        "epochs": 10,
        "trainable_backbone_layers": 3,
        "optimizer": "SGD",
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "batch_size": 8,
        "augment": True,
        "seed": 42,
        "start_checkpoint": "fasterrcnn_v2_freeze_head/weights/best.pt",
    })

    try: 
        train_detection_run(model, optimizer, train_loader, val_loader, device, run_dir, epochs=10)
        df = pd.read_csv(run_dir / "metrics.csv")
        best = df.loc[df["map50"].idxmax()]
        results.append({
            "run": run_dir.name,
            "lr": lr,
            "best_epoch": int(best["epoch"]),
            "map50": float(best["map50"]),
            "map75": float(best["map75"]),
            "ap50_fire": float(best["ap50_fire"]),
            "ap50_smoke": float(best["ap50_smoke"]),
            "val_total": float(best["val_total"]),
        })
    except Exception as e:
        print(f"Training failed for {lr}: {e}")

Epoch 1, Step 200, Loss: 0.10873074986040593
Epoch 1, Step 400, Loss: 0.10835421157069504
AP@0.50  fire: 0.955
AP@0.50  smoke: 0.860
mAP@0.50: 0.908
AP@0.75  fire: 0.700
AP@0.75  smoke: 0.446
mAP@0.75: 0.573
Epoch 1: val_total 0.11502152827619774, map50 0.9077920222603026, map75 0.5732109419253593, 177.37362360954285s
Epoch 2, Step 200, Loss: 0.10771121121942998
Epoch 2, Step 400, Loss: 0.10505520619452
AP@0.50  fire: 0.956
AP@0.50  smoke: 0.868
mAP@0.50: 0.912
AP@0.75  fire: 0.732
AP@0.75  smoke: 0.501
mAP@0.75: 0.617
Epoch 2: val_total 0.1127033107028417, map50 0.9121643366559188, map75 0.6167833834558395, 175.66290616989136s
Epoch 3, Step 200, Loss: 0.0995040369592607
Epoch 3, Step 400, Loss: 0.10056867348961532
AP@0.50  fire: 0.954
AP@0.50  smoke: 0.868
mAP@0.50: 0.911
AP@0.75  fire: 0.749
AP@0.75  smoke: 0.494
mAP@0.75: 0.622
Epoch 3: val_total 0.11199912412857717, map50 0.9111306421142571, map75 0.6217898719058004, 175.64452147483826s
Epoch 4, Step 200, Loss: 0.09784748289734126


In [ ]:
summary = pd.DataFrame(results)
summary.to_csv(RUNS / "lr_sweep_e10_summary.csv", index=False)
summary